In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("Incremental Flag","0")

In [0]:
increm_flag=dbutils.widgets.get("Incremental Flag")
print(increm_flag)

In [0]:
df_enc_src=spark.sql('''
                     select * from  parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/encounters`''')
df_enc_src.limit(10).display()

In [0]:
df_encounters=spark.sql('''
            select encounter_type,admit_source,discharge_disposition,length_of_stay_days,total_charge_amount,readmission_30day_flag,source_system,_ingest_timestamp from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/encounters`''')

In [0]:
df_encounters.limit(5).display()

In [0]:
df_dim_patient=spark.sql("SELECT * FROM healthcare.gold.dim_patient")
df_dim_date=spark.sql("SELECT * FROM healthcare.gold.dim_date")
df_dim_facility=spark.sql("SELECT * FROM healthcare.gold.dim_facility")
df_dim_payer=spark.sql("SELECT * FROM healthcare.gold.dim_payer")
df_dim_diagnosis=spark.sql("SELECT * FROM healthcare.gold.dim_diagnosis")
df_dim_provider=spark.sql("SELECT * FROM healthcare.gold.dim_provider")

In [0]:
df_enc_src.count()

In [0]:
df_dim_patient.count()

In [0]:
df_dim_facility.count()

In [0]:
df_staged_fact = (
    df_enc_src.join(
        df_dim_patient,
        ((df_enc_src["mrn"] == df_dim_patient["mrn"]) 
         #we are commenting out the date filter as we are working with sample data and not enough data is present
         & (df_enc_src["admit_date"].between(df_dim_patient["effective_start_date"], df_dim_patient["effective_end_date"]))
         ),
        "inner",
    )
    .join(
        df_dim_provider,
        ((df_enc_src["rendering_npi"] == df_dim_provider["npi"]) 
         & (df_enc_src["admit_date"].between(df_dim_provider["effective_start_date"], df_dim_provider["effective_end_date"]))
         ),
        "inner",
    )
    .join(df_dim_facility, df_enc_src["facility_id"] == df_dim_facility["facility_id"], "inner")
    .join(df_dim_payer, df_enc_src["payer_id"] == df_dim_payer["payer_id"], "inner")
    .join(df_dim_date, df_enc_src["admit_date"] == df_dim_date["full_date"], "inner")
    .select(
        df_enc_src["encounter_type"],
        df_enc_src["encounter_id"],
        df_enc_src["admit_source"],
        df_enc_src["discharge_disposition"],
        df_enc_src["length_of_stay_days"],
        df_enc_src["total_charge_amount"],
        df_enc_src["readmission_30day_flag"],
        df_enc_src["source_system"],
        df_enc_src["_ingest_timestamp"],
        df_dim_patient["dim_patient_key"],
        df_dim_provider["dim_provider_key"],
        df_dim_facility["dim_facility_key"],
        df_dim_payer["dim_payer_key"],
        df_dim_date["dim_date_key"],
    )
)

In [0]:
df_staged_fact.display()

In [0]:
df_staged_fact.count()

In [0]:
if increm_flag=='0':
    max_value=1
else:
    max_value_df = spark.sql('''
    SELECT coalesce(MAX(encounter_key), 0) FROM healthcare.gold.fact_encounter
''')
    max_value = max_value_df.collect()[0][0] + 1

In [0]:
if spark.catalog.tableExists("healthcare.gold.fact_encounter"):
    df_sink=spark.sql('''
                      select encounter_key,encounter_id from healthcare.gold.fact_encounter''')
else:
    df_sink=spark.sql('''
                      select 1 as encounter_key,encounter_id from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/encounters` where 1=0''')

In [0]:
df_sink.limit(5).display()

In [0]:
df_filter=df_staged_fact.join(df_sink,df_staged_fact["encounter_id"]==df_sink["encounter_id"],"left").select(df_staged_fact["*"],df_sink["encounter_key"])

In [0]:
df_filter.limit(5).display()

In [0]:
df_new_rec=df_filter.filter(df_filter["encounter_key"].isNull())
df_existing_rec=df_filter.filter(df_filter["encounter_key"].isNotNull())
df_new_rec.limit(5).display()
df_existing_rec.limit(5).display()

In [0]:
windowSpec = Window.orderBy("encounter_type")
df_insert = df_new_rec.withColumn(
    "encounter_key", row_number().over(windowSpec) + lit(max_value - 1)
)

In [0]:
df_insert.limit(5).display()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.fact_encounter"):
    #insert the new entries
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_encounter")
    #update the entries
    deltaTable=DeltaTable.forName(spark,"healthcare.gold.fact_encounter")
    deltaTable.alias("target").merge(df_existing_rec.alias("source"),"target.encounter_key = source.encounter_key")\
        .whenMatchedUpdate(set={
            "dim_patient_key": "source.dim_patient_key",
            "dim_provider_key": "source.dim_provider_key",
            "dim_facility_key": "source.dim_facility_key",
            "dim_payer_key": "source.dim_payer_key",
            "dim_date_key": "source.dim_date_key",
            "encounter_id": "source.encounter_id",
    "encounter_type":"source.encounter_type",
    "admit_source":"source.admit_source",
    "readmission_30day_flag" :"source.readmission_30day_flag",
    "discharge_disposition": "source.discharge_disposition",
    "total_charge_amount": "source.total_charge_amount",
    "source_system": "source.source_system",
    "length_of_stay_days": "source.length_of_stay_days",
    }).execute()
else:
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.fact_encounter")

In [0]:
%sql
select * from healthcare.gold.fact_encounter;